<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


This edition uses `LiquidAI/LFM3-1.7B-Base` as the trainable candidate. The `reasoning` loader option changes prompt formatting while keeping the same base weights. Train and evaluate new LFM3 checkpoints with the chapter code. Original Qwen benchmark results and training logs are historical references, not LFM3 measurements.

# Chapter 7: Exercise Solutions

Packages that are being used in this notebook:

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## Exercise 7.1: Testing the `<think>` format reward

- The following code checks that the format reward is zero if the think tokens are used incorrectly

In [ ]:
from pathlib import Path
from reasoning_from_scratch.lfm3 import LFM3Tokenizer
from reasoning_from_scratch.lfm3 import download_lfm3

download_lfm3(
    kind="reasoning", tokenizer_only=True, out_dir="lfm3"
)

tokenizer_path = Path("lfm3") / "tokenizer.json"
tokenizer = LFM3Tokenizer(tokenizer_file_path=tokenizer_path)

In [ ]:
import torch
from reasoning_from_scratch.ch07 import reward_format

prompt = "Calculate ..."

def check_case(name, rollout):
    token_ids = tokenizer.encode(prompt + rollout)
    prompt_len = len(tokenizer.encode(prompt))

    reward = reward_format(
        token_ids=torch.tensor(token_ids),
        prompt_len=prompt_len,
    )

    print(f"{name}: {reward}")


# 1) Correct case
check_case(
    "Correct order",
    "Let's ... <think> ... </think> ..."
)

# 2) Typo in tag
check_case(
    "Typo in <think>",
    "Let's ... <think_missing> ... </think> ..."
)

# 3) Reversed order
check_case(
    "Reversed order",
    "Let's ... </think> ... <think> ..."
)

# 4) Missing one tag
check_case(
    "Missing </think>",
    "Let's ... <think> ..."
)

- The output indicates that the function requires correct <think>...</think> tag use to award a reward of 1.0

&nbsp;
## Exercise 7.2: Conditional format rewards

- The implementation of the conditional reward is very simple; in the main chapter, we discussed implementing the overall reward as follows:

```python
reward = rlvr_reward + format_reward_weight * format_reward
```

- So, one way to disable the reward if the correctness reward (`rlvr_reward`) is 0.0 is

```python
if conditional_reward:
    format_reward *= rlvr_reward
reward = rlvr_reward + format_reward_weight * format_reward
```

- To use it in practice, you can run the [03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py](../03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py) script (similar to section 7.6 in chapter 7) with the `--conditional_reward` flag enabled
- To analyze the results, let's first plot the original results from chapter 7.6 again without conditional rewards for reference:

In [ ]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch07/02_logs/7_6_plus_format_reward_metrics.csv"
)

In [ ]:
from reasoning_from_scratch.ch07 import plot_grpo_metrics

plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
)

In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["reward_avg", "format_reward_avg", "adv_std", "entropy_avg"],
)

- Run a conditional-reward experiment and set the CSV path below to its metrics file. The upstream repository does not publish the conditional-run CSV.

In [ ]:
from pathlib import Path

conditional_csv = Path("7_6_plus_format_reward_conditional_metrics.csv")
if not conditional_csv.exists():
    print("Set conditional_csv to your conditional-reward training log.")

In [ ]:
if conditional_csv.exists():
    plot_grpo_metrics(
        conditional_csv,
        columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    )

- Inspect whether accuracy and correctness rewards recover after early training changes. Draw conclusions from your own run; these plots do not contain LFM3 results until you train it.

In [ ]:
if conditional_csv.exists():
    plot_grpo_metrics(
        conditional_csv,
        columns=["reward_avg", "format_reward_avg", "adv_std", "entropy_avg"],
    )

- Compare format rewards on correct rollouts with overall correctness rewards. Check entropy as well as reward trends when assessing stability.